# Regression: Model Snippets (from scratch, ก้อปแล้วใช้ได้ทันที)

แต่ละ cell **ครบในตัวเอง** (รวมคลาสที่ต้องพึ่งพาไว้แล้ว เช่น Random Forest มี Decision Tree ติดมาด้วย) ก้อปทั้ง cell ไปวางแทน cell `MODEL SLOT` ใน `regression_submission_skeleton.ipynb`

| โมเดล | `FEATURES` | พึ่งพาคลาสอื่น (ติดมาในcell แล้ว) |
|---|---|---|
| Linear / Ridge | scaled | - |
| Gradient Descent | scaled | - |
| Lasso | scaled | - |
| Poly2 + Ridge | scaled | LinearRegressionScratch |
| KNN | scaled | - |
| Decision Tree | raw | - |
| Random Forest | raw | DecisionTreeScratch |
| Gradient Boosting | raw | DecisionTreeScratch |

ค่า hyperparameter ใน `make_model()` เป็นค่าเริ่มต้น ให้แก้เป็นค่าที่จูนได้จาก `regression_exam.ipynb`

(โมเดลที่ `scaled` คือข้อมูลที่ standardize แล้ว ส่วน tree-based ใช้ `raw` ได้ผลเท่ากัน)

### Linear Regression (Normal Equation)

In [ ]:
# ===== MODEL: Linear Regression (Normal Equation) =====
# วิธีใช้: ก้อปทั้ง cell นี้ไปวางแทน cell "MODEL SLOT" ใน skeleton (ไม่ต้องแก้อย่างอื่น)
import numpy as np

class LinearRegressionScratch:
    def __init__(self, lam=0.0):
        self.lam = lam

    def fit(self, X, y):
        A = np.c_[np.ones(len(X)), X]               # เติมคอลัมน์ 1 = intercept
        I = np.eye(A.shape[1]); I[0, 0] = 0         # ไม่ penalize intercept
        self.w = np.linalg.pinv(A.T @ A + self.lam * I) @ A.T @ y
        return self

    def predict(self, X):
        return np.c_[np.ones(len(X)), X] @ self.w

FEATURES = 'scaled'     # 'scaled' = ข้อมูลที่ standardize แล้ว | 'raw' = ข้อมูลดิบ

def make_model():
    return LinearRegressionScratch(lam=0.0)      # <-- แก้ค่า hyperparameter ตามที่จูนได้

### Ridge Regression

In [ ]:
# ===== MODEL: Ridge Regression =====
# วิธีใช้: ก้อปทั้ง cell นี้ไปวางแทน cell "MODEL SLOT" ใน skeleton (ไม่ต้องแก้อย่างอื่น)
import numpy as np

class LinearRegressionScratch:
    def __init__(self, lam=0.0):
        self.lam = lam

    def fit(self, X, y):
        A = np.c_[np.ones(len(X)), X]               # เติมคอลัมน์ 1 = intercept
        I = np.eye(A.shape[1]); I[0, 0] = 0         # ไม่ penalize intercept
        self.w = np.linalg.pinv(A.T @ A + self.lam * I) @ A.T @ y
        return self

    def predict(self, X):
        return np.c_[np.ones(len(X)), X] @ self.w

FEATURES = 'scaled'     # 'scaled' = ข้อมูลที่ standardize แล้ว | 'raw' = ข้อมูลดิบ

def make_model():
    return LinearRegressionScratch(lam=10.0)      # <-- แก้ค่า hyperparameter ตามที่จูนได้   (lam: ค่ามาก = regularize แรง (ลอง 0.1, 1, 10, 100))

### Linear Regression (Gradient Descent)

In [ ]:
# ===== MODEL: Linear Regression (Gradient Descent) =====
# วิธีใช้: ก้อปทั้ง cell นี้ไปวางแทน cell "MODEL SLOT" ใน skeleton (ไม่ต้องแก้อย่างอื่น)
import numpy as np

class GDRegression:
    def __init__(self, lr=0.05, epochs=2000, lam=0.0):
        self.lr, self.epochs, self.lam = lr, epochs, lam

    def fit(self, X, y):
        n, d = X.shape
        self.w, self.b = np.zeros(d), 0.0
        self.loss_history = []
        for _ in range(self.epochs):
            err = X @ self.w + self.b - y
            self.w -= self.lr * ((2 / n) * X.T @ err + 2 * self.lam * self.w)
            self.b -= self.lr * (2 / n) * err.sum()
            self.loss_history.append(np.mean(err ** 2))
        return self

    def predict(self, X):
        return X @ self.w + self.b

FEATURES = 'scaled'     # 'scaled' = ข้อมูลที่ standardize แล้ว | 'raw' = ข้อมูลดิบ

def make_model():
    return GDRegression(lr=0.05, epochs=2000)      # <-- แก้ค่า hyperparameter ตามที่จูนได้   (ถ้าได้ nan ให้ลด lr เป็น 0.01)

### Lasso (Coordinate Descent)

In [ ]:
# ===== MODEL: Lasso (Coordinate Descent) =====
# วิธีใช้: ก้อปทั้ง cell นี้ไปวางแทน cell "MODEL SLOT" ใน skeleton (ไม่ต้องแก้อย่างอื่น)
import numpy as np

class LassoScratch:
    def __init__(self, alpha=0.1, n_iter=1000, tol=1e-6):
        self.alpha, self.n_iter, self.tol = alpha, n_iter, tol

    def fit(self, X, y):                      # X ต้อง standardize แล้ว
        n, d = X.shape
        self.b = y.mean()
        self.w = np.zeros(d)
        r = y - self.b                        # residual
        z = (X ** 2).sum(axis=0) / n
        for _ in range(self.n_iter):
            w_old = self.w.copy()
            for j in range(d):
                if z[j] == 0: continue
                r += X[:, j] * self.w[j]                   # เอาผลของ feature j ออก
                rho = X[:, j] @ r / n
                self.w[j] = np.sign(rho) * max(abs(rho) - self.alpha, 0) / z[j]   # soft-threshold
                r -= X[:, j] * self.w[j]
            if np.max(np.abs(self.w - w_old)) < self.tol:
                break
        return self

    def predict(self, X):
        return X @ self.w + self.b

FEATURES = 'scaled'     # 'scaled' = ข้อมูลที่ standardize แล้ว | 'raw' = ข้อมูลดิบ

def make_model():
    return LassoScratch(alpha=0.1)      # <-- แก้ค่า hyperparameter ตามที่จูนได้   (alpha: ค่ามาก = ตัด feature มาก)

### Polynomial (degree 2) + Ridge

In [ ]:
# ===== MODEL: Polynomial (degree 2) + Ridge =====
# วิธีใช้: ก้อปทั้ง cell นี้ไปวางแทน cell "MODEL SLOT" ใน skeleton (ไม่ต้องแก้อย่างอื่น)
import numpy as np

class LinearRegressionScratch:
    def __init__(self, lam=0.0):
        self.lam = lam

    def fit(self, X, y):
        A = np.c_[np.ones(len(X)), X]               # เติมคอลัมน์ 1 = intercept
        I = np.eye(A.shape[1]); I[0, 0] = 0         # ไม่ penalize intercept
        self.w = np.linalg.pinv(A.T @ A + self.lam * I) @ A.T @ y
        return self

    def predict(self, X):
        return np.c_[np.ones(len(X)), X] @ self.w

class PolyRidgeScratch:
    def __init__(self, lam=10.0):
        self.lam = lam

    def _poly(self, X):
        i, j = np.triu_indices(X.shape[1])               # ทุกคู่ x_i * x_j (รวมกำลังสอง)
        return np.hstack([X, X[:, i] * X[:, j]])

    def fit(self, X, y):
        P = self._poly(X)
        self.pm, self.ps = P.mean(axis=0), P.std(axis=0)
        self.ps[self.ps == 0] = 1
        self.m = LinearRegressionScratch(self.lam).fit((P - self.pm) / self.ps, y)   # (ใช้คลาสด้านบน)
        return self

    def predict(self, X):
        return self.m.predict((self._poly(X) - self.pm) / self.ps)

FEATURES = 'scaled'     # 'scaled' = ข้อมูลที่ standardize แล้ว | 'raw' = ข้อมูลดิบ

def make_model():
    return PolyRidgeScratch(lam=10.0)      # <-- แก้ค่า hyperparameter ตามที่จูนได้   (ถ้า feature เยอะ (>40) จะช้า/overfit)

### KNN Regression

In [ ]:
# ===== MODEL: KNN Regression =====
# วิธีใช้: ก้อปทั้ง cell นี้ไปวางแทน cell "MODEL SLOT" ใน skeleton (ไม่ต้องแก้อย่างอื่น)
import numpy as np

class KNNScratch:
    def __init__(self, k=5, weighted=False):
        self.k, self.weighted = k, weighted

    def fit(self, X, y):
        self.X, self.y = X, y
        return self

    def predict(self, X):
        out = []
        for s in range(0, len(X), 500):                    # แบ่งเป็นก้อน กัน memory เต็ม
            Q = X[s:s + 500]
            d2 = (Q**2).sum(1)[:, None] + (self.X**2).sum(1)[None, :] - 2 * Q @ self.X.T
            d2 = np.maximum(d2, 0)
            k = min(self.k, self.X.shape[0])
            idx = np.argpartition(d2, k - 1, axis=1)[:, :k]   # เพื่อนบ้านที่ใกล้สุด k ตัว
            if not self.weighted:
                out.append(self.y[idx].mean(axis=1))
            else:
                w = 1 / (np.sqrt(np.take_along_axis(d2, idx, axis=1)) + 1e-9)
                out.append((self.y[idx] * w).sum(axis=1) / w.sum(axis=1))
        return np.concatenate(out)

FEATURES = 'scaled'     # 'scaled' = ข้อมูลที่ standardize แล้ว | 'raw' = ข้อมูลดิบ

def make_model():
    return KNNScratch(k=5, weighted=False)      # <-- แก้ค่า hyperparameter ตามที่จูนได้   (k: ลอง 3, 5, 10, 15, 25)

### Decision Tree

In [ ]:
# ===== MODEL: Decision Tree =====
# วิธีใช้: ก้อปทั้ง cell นี้ไปวางแทน cell "MODEL SLOT" ใน skeleton (ไม่ต้องแก้อย่างอื่น)
import numpy as np

class DecisionTreeScratch:
    def __init__(self, max_depth=5, min_samples_leaf=5, max_features=None, rng=None):
        self.max_depth, self.msl = max_depth, min_samples_leaf
        self.max_features = max_features                   # None = ใช้ทุก feature
        self.rng = rng if rng is not None else np.random.default_rng(42)

    def fit(self, X, y):
        self.tree = self._build(X, y, 0)
        return self

    def _best_split(self, X, y):
        n, d = X.shape
        feats = np.arange(d)
        if self.max_features is not None:
            feats = self.rng.choice(d, size=min(self.max_features, d), replace=False)
        best_j, best_t, best_sse = None, None, np.inf
        for j in feats:
            order = np.argsort(X[:, j])
            xs, ys = X[order, j], y[order]
            cs, cs2 = np.cumsum(ys), np.cumsum(ys ** 2)
            nl = np.arange(self.msl, n - self.msl + 1)     # จำนวนข้อมูลฝั่งซ้ายที่เป็นไปได้
            if len(nl) == 0: continue
            sl, sl2 = cs[nl - 1], cs2[nl - 1]
            nr, sr, sr2 = n - nl, cs[-1] - sl, cs2[-1] - sl2
            sse = (sl2 - sl**2 / nl) + (sr2 - sr**2 / nr)  # SSE ซ้าย + ขวา
            sse[xs[nl - 1] == xs[nl]] = np.inf             # ห้ามตัดระหว่างค่าที่เท่ากัน
            k = np.argmin(sse)
            if sse[k] < best_sse:
                best_sse, best_j = sse[k], j
                best_t = (xs[nl[k] - 1] + xs[nl[k]]) / 2
        return best_j, best_t

    def _build(self, X, y, depth):
        if depth >= self.max_depth or len(y) < 2 * self.msl or np.all(y == y[0]):
            return {'value': y.mean()}
        j, t = self._best_split(X, y)
        if j is None:
            return {'value': y.mean()}
        mask = X[:, j] <= t
        return {'j': j, 't': t,
                'left':  self._build(X[mask],  y[mask],  depth + 1),
                'right': self._build(X[~mask], y[~mask], depth + 1)}

    def _predict_one(self, x):
        node = self.tree
        while 'value' not in node:
            node = node['left'] if x[node['j']] <= node['t'] else node['right']
        return node['value']

    def predict(self, X):
        return np.array([self._predict_one(x) for x in X])

FEATURES = 'raw'     # 'scaled' = ข้อมูลที่ standardize แล้ว | 'raw' = ข้อมูลดิบ

def make_model():
    return DecisionTreeScratch(max_depth=5, min_samples_leaf=5)      # <-- แก้ค่า hyperparameter ตามที่จูนได้   (max_depth ลึก = overfit ง่าย)

### Random Forest

In [ ]:
# ===== MODEL: Random Forest =====
# วิธีใช้: ก้อปทั้ง cell นี้ไปวางแทน cell "MODEL SLOT" ใน skeleton (ไม่ต้องแก้อย่างอื่น)
import numpy as np

class DecisionTreeScratch:
    def __init__(self, max_depth=5, min_samples_leaf=5, max_features=None, rng=None):
        self.max_depth, self.msl = max_depth, min_samples_leaf
        self.max_features = max_features                   # None = ใช้ทุก feature
        self.rng = rng if rng is not None else np.random.default_rng(42)

    def fit(self, X, y):
        self.tree = self._build(X, y, 0)
        return self

    def _best_split(self, X, y):
        n, d = X.shape
        feats = np.arange(d)
        if self.max_features is not None:
            feats = self.rng.choice(d, size=min(self.max_features, d), replace=False)
        best_j, best_t, best_sse = None, None, np.inf
        for j in feats:
            order = np.argsort(X[:, j])
            xs, ys = X[order, j], y[order]
            cs, cs2 = np.cumsum(ys), np.cumsum(ys ** 2)
            nl = np.arange(self.msl, n - self.msl + 1)     # จำนวนข้อมูลฝั่งซ้ายที่เป็นไปได้
            if len(nl) == 0: continue
            sl, sl2 = cs[nl - 1], cs2[nl - 1]
            nr, sr, sr2 = n - nl, cs[-1] - sl, cs2[-1] - sl2
            sse = (sl2 - sl**2 / nl) + (sr2 - sr**2 / nr)  # SSE ซ้าย + ขวา
            sse[xs[nl - 1] == xs[nl]] = np.inf             # ห้ามตัดระหว่างค่าที่เท่ากัน
            k = np.argmin(sse)
            if sse[k] < best_sse:
                best_sse, best_j = sse[k], j
                best_t = (xs[nl[k] - 1] + xs[nl[k]]) / 2
        return best_j, best_t

    def _build(self, X, y, depth):
        if depth >= self.max_depth or len(y) < 2 * self.msl or np.all(y == y[0]):
            return {'value': y.mean()}
        j, t = self._best_split(X, y)
        if j is None:
            return {'value': y.mean()}
        mask = X[:, j] <= t
        return {'j': j, 't': t,
                'left':  self._build(X[mask],  y[mask],  depth + 1),
                'right': self._build(X[~mask], y[~mask], depth + 1)}

    def _predict_one(self, x):
        node = self.tree
        while 'value' not in node:
            node = node['left'] if x[node['j']] <= node['t'] else node['right']
        return node['value']

    def predict(self, X):
        return np.array([self._predict_one(x) for x in X])

class RandomForestScratch:
    def __init__(self, n_trees=50, max_depth=8, min_samples_leaf=3, max_features=None, seed=42):
        self.n_trees, self.max_depth, self.msl = n_trees, max_depth, min_samples_leaf
        self.max_features, self.seed = max_features, seed

    def fit(self, X, y):
        rng = np.random.default_rng(self.seed)
        n, d = X.shape
        mf = self.max_features or max(1, d // 3)           # สุ่มใช้ ~1/3 ของ feature ในแต่ละการแยก
        self.trees = []
        for _ in range(self.n_trees):
            idx = rng.integers(0, n, n)                    # bootstrap sample
            t = DecisionTreeScratch(self.max_depth, self.msl, max_features=mf, rng=rng)
            self.trees.append(t.fit(X[idx], y[idx]))
        return self

    def predict(self, X):
        return np.mean([t.predict(X) for t in self.trees], axis=0)

FEATURES = 'raw'     # 'scaled' = ข้อมูลที่ standardize แล้ว | 'raw' = ข้อมูลดิบ

def make_model():
    return RandomForestScratch(n_trees=50, max_depth=8, min_samples_leaf=3)      # <-- แก้ค่า hyperparameter ตามที่จูนได้   (ถ้าช้า ลด n_trees เหลือ 20-30)

### Gradient Boosting

In [ ]:
# ===== MODEL: Gradient Boosting =====
# วิธีใช้: ก้อปทั้ง cell นี้ไปวางแทน cell "MODEL SLOT" ใน skeleton (ไม่ต้องแก้อย่างอื่น)
import numpy as np

class DecisionTreeScratch:
    def __init__(self, max_depth=5, min_samples_leaf=5, max_features=None, rng=None):
        self.max_depth, self.msl = max_depth, min_samples_leaf
        self.max_features = max_features                   # None = ใช้ทุก feature
        self.rng = rng if rng is not None else np.random.default_rng(42)

    def fit(self, X, y):
        self.tree = self._build(X, y, 0)
        return self

    def _best_split(self, X, y):
        n, d = X.shape
        feats = np.arange(d)
        if self.max_features is not None:
            feats = self.rng.choice(d, size=min(self.max_features, d), replace=False)
        best_j, best_t, best_sse = None, None, np.inf
        for j in feats:
            order = np.argsort(X[:, j])
            xs, ys = X[order, j], y[order]
            cs, cs2 = np.cumsum(ys), np.cumsum(ys ** 2)
            nl = np.arange(self.msl, n - self.msl + 1)     # จำนวนข้อมูลฝั่งซ้ายที่เป็นไปได้
            if len(nl) == 0: continue
            sl, sl2 = cs[nl - 1], cs2[nl - 1]
            nr, sr, sr2 = n - nl, cs[-1] - sl, cs2[-1] - sl2
            sse = (sl2 - sl**2 / nl) + (sr2 - sr**2 / nr)  # SSE ซ้าย + ขวา
            sse[xs[nl - 1] == xs[nl]] = np.inf             # ห้ามตัดระหว่างค่าที่เท่ากัน
            k = np.argmin(sse)
            if sse[k] < best_sse:
                best_sse, best_j = sse[k], j
                best_t = (xs[nl[k] - 1] + xs[nl[k]]) / 2
        return best_j, best_t

    def _build(self, X, y, depth):
        if depth >= self.max_depth or len(y) < 2 * self.msl or np.all(y == y[0]):
            return {'value': y.mean()}
        j, t = self._best_split(X, y)
        if j is None:
            return {'value': y.mean()}
        mask = X[:, j] <= t
        return {'j': j, 't': t,
                'left':  self._build(X[mask],  y[mask],  depth + 1),
                'right': self._build(X[~mask], y[~mask], depth + 1)}

    def _predict_one(self, x):
        node = self.tree
        while 'value' not in node:
            node = node['left'] if x[node['j']] <= node['t'] else node['right']
        return node['value']

    def predict(self, X):
        return np.array([self._predict_one(x) for x in X])

class GradientBoostingScratch:
    def __init__(self, n_estimators=100, lr=0.1, max_depth=3, min_samples_leaf=5):
        self.n, self.lr, self.max_depth, self.msl = n_estimators, lr, max_depth, min_samples_leaf

    def fit(self, X, y):
        self.base = y.mean()
        pred = np.full(len(y), self.base)
        self.trees = []
        for _ in range(self.n):
            resid = y - pred                               # fit ต้นไม้เล็กๆ กับ residual
            t = DecisionTreeScratch(self.max_depth, self.msl).fit(X, resid)
            pred += self.lr * t.predict(X)
            self.trees.append(t)
        return self

    def predict(self, X):
        pred = np.full(len(X), self.base)
        for t in self.trees:
            pred += self.lr * t.predict(X)
        return pred

FEATURES = 'raw'     # 'scaled' = ข้อมูลที่ standardize แล้ว | 'raw' = ข้อมูลดิบ

def make_model():
    return GradientBoostingScratch(n_estimators=100, lr=0.1, max_depth=3)      # <-- แก้ค่า hyperparameter ตามที่จูนได้   (n_estimators มาก + lr น้อย มักดีแต่ช้า)